### Zusammenhangskomponenten finden
Nutzen `matrix_helpers` und 
einen Algorithmus zu implementieren, der alle 
Felder eines Gitters einer Farbe findet, die eine Seite/ mind. eine Ecke gemeinsam haben.
Wir nennen solche Felder gleichfarbige Nachbarn.

Die implementation benutzt in typischer Weise einen Liste `stack`, die
alle Felder enthält, deren gleichfarbige Nachbarn wir noch nicht bestimmt haben.
Das Resultat ist eine Menge `component`.



Menge, das rascher als mit liste testbar, ob element bereits enthalten.
- pfad finden
- minen raeumen: benachbart umdefinieren:
  pos is nachbar von pos0, falls gemeinsame Eckepos (und Minenfeld) und pos0 hat null minencount 0
  bez. felder mit pos minencount haben keine nachbarn.
- benutze zum floodfill: finde z.B> weisse felder. Statt componente zurueckzugeben, faerbe sie um
- pfad finden


In [2]:
import random
import matrix_helpers as M


N = 10
matrix = M.make_matrix(N)
for i in range(N*N):
    M.set_item(matrix, M.idx2pos(i, ncol=N), random.randint(0, 2))
M.show_matrix(matrix)

2,1,2,2,2,0,0,2,1,1
0,1,0,0,2,1,2,2,1,2
0,1,2,2,2,2,1,0,1,2
2,0,2,2,0,2,1,2,1,0
0,0,2,1,1,1,0,0,1,2
2,2,1,2,1,1,2,2,2,1
1,0,0,1,1,2,1,1,0,1
2,2,0,2,1,1,2,2,2,0
1,2,0,1,2,2,2,2,1,0
0,1,0,1,0,1,2,0,2,2


In [14]:
def get_component(matrix, start, kinds='sc'):
    component = {start}
    stack = [start]
    while stack:
        pos0 = stack.pop()
        for pos in M.get_neighbors(matrix, pos0, kinds=kinds):
            if (M.get_cell(matrix, pos0) == M.get_cell(matrix, pos)
                    and pos not in component):
                component.add(pos)
                stack.append(pos)
    return component

In [15]:
get_component(matrix, start=(0, 0))

{(0, 0), (1, 0)}

In [4]:
def draw_matrix(canvas, matrix, grid_spec, cd):
    for row, cols in enumerate(matrix):
        for col, val in enumerate(cols):
            pos = (col, row)
            G.fill_rect(bg, pos, grid_spec, color=cd[val])

In [5]:
import widget_helpers as W
import grid_helpers as G


grid_spec = G.make_grid_spec(ncol=10)

mcanvas = W.get_mcanvas(3)
bg, mg, fg = mcanvas
mg.global_alpha = 0.4
mg.fill_style = 'white'

G.draw_grid(fg, grid_spec)
colordict = {0: 'red', 1: 'green', 2: 'blue'}
draw_matrix(bg, matrix, grid_spec, colordict)


mcanvas

MultiCanvas(height=100, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_r…

In [6]:
def mark_component(matrix, pos, kinds='sc'):
    comp = get_component(matrix, pos, kinds=kinds)
    mg.clear()
    for pos in comp:
        G.fill_rect(mg, pos, grid_spec)

In [7]:
mark_component(matrix, (0, 0))

In [8]:
def on_mouse_down(x, y):
    pos = G.xy2cr(x, y, grid_spec)
    mark_component(matrix, pos)

In [9]:
on_mouse_down(5, 25)

In [10]:
mcanvas.on_mouse_down(on_mouse_down)

In [11]:
matrix = [
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 1, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 1, 1, 1, 1, 0, 0, 1, 0],
 [0, 0, 0, 0, 0, 1, 0, 1, 1, 0],
 [1, 0, 0, 0, 0, 1, 0, 0, 1, 0],
 [1, 0, 1, 1, 1, 1, 0, 0, 1, 0],
 [1, 0, 1, 0, 0, 0, 0, 0, 1, 0],
 [1, 0, 1, 0, 1, 1, 1, 1, 1, 0],
 [1, 0, 1, 1, 1, 0, 0, 0, 0, 0],
 [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
]


mg.clear()
bg.clear()
draw_matrix(bg, matrix, grid_spec, {0: 'grey', 1: 'blue'})
mcanvas

MultiCanvas(height=100, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_r…

In [12]:
matrix = [
 [0, 0, 1, 1, 1, 0, 0, 0, 1, 1],
 [1, 1, 2, 9, 2, 0, 0, 0, 1, 9],
 [9, 1, 2, 9, 3, 1, 1, 0, 1, 1],
 [1, 1, 1, 1, 2, 9, 1, 0, 0, 0],
 [0, 0, 0, 0, 1, 1, 1, 1, 1, 1],
 [0, 0, 0, 0, 0, 0, 0, 2, 9, 2],
 [0, 0, 0, 0, 0, 1, 2, 4, 9, 2],
 [1, 1, 1, 0, 0, 1, 9, 9, 2, 1],
 [1, 9, 1, 0, 0, 1, 2, 2, 1, 0],
 [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
]


mg.clear()
bg.clear()
colordict = {0: 'grey', 1: 'green', 2: 'orange', 3: 'red', 4: 'blue', 9: 'black'}
draw_matrix(bg, matrix, grid_spec, colordict)
mcanvas

MultiCanvas(height=100, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_r…

In [13]:
def get_component_1(matrix, start, get_neighbors):
    component = {start}
    stack = [start]
    while stack:
        pos0 = stack.pop()
        for pos in get_neighbors(matrix, pos0):
            if pos not in component:
                component.add(pos)
                stack.append(pos)
    return component

In [ ]:
def get_neighbors(matrix, pos):
    if M.get_item()
    ns = M.get_neighbors(matrix, pos, kinds='s')
    return [pos for pos in ns if M.get_cell(pos) == 9]